## Cross-Dataset Generalization, Language-wise Robustness, Ablation, Error & Transfer Analysis

## 1. Setup & Install

In [ ]:
!pip install -q transformers scikit-learn matplotlib seaborn scipy statsmodels openpyxl pyarabic
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%matplotlib inline
import os, re, json, random, time, pickle, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, confusion_matrix,
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    matthews_corrcoef, cohen_kappa_score, roc_curve, precision_recall_curve
)
from statsmodels.stats.proportion import proportion_confint
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import joblib, pyarabic.araby as araby

print("Imports OK. CUDA available:", torch.cuda.is_available())


## 2. Config — all paths in one place

In [ ]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

#  Datasets 
DATASETS_BASE = "/content/drive/MyDrive/Datasets"
AAAC_DIR   = f"{DATASETS_BASE}/AAAC"
OSACT_DIR  = f"{DATASETS_BASE}/OSACT"
LAYER1_DIR = f"{DATASETS_BASE}/layer1"

AAAC_TRAIN_PATH  = f"{AAAC_DIR}/train_aaac.csv"
AAAC_VAL_PATH    = f"{AAAC_DIR}/val_aaac.csv"
AAAC_TEST_PATH   = f"{AAAC_DIR}/test_aaac.csv"
OSACT_TRAIN_PATH = f"{OSACT_DIR}/train_osact.csv"
OSACT_DEV_PATH   = f"{OSACT_DIR}/dev_osact.csv"
LAYER1_TRAIN_PATH = f"{LAYER1_DIR}/layer1_train.csv"
LAYER1_VAL_PATH   = f"{LAYER1_DIR}/layer1_validation.csv"
LAYER1_TEST_PATH  = f"{LAYER1_DIR}/layer1_test.csv"

#  04a outputs (results/predictions/models produced by 04a) 
OUTPUT_DIR  = f"{LAYER1_DIR}/outputs"
MODELS_DIR  = f"{OUTPUT_DIR}/models"          # MODELS_DIR/{osact,unified,aaac_transfer}/{model_key}
RESULTS_DIR = f"{OUTPUT_DIR}/results"         # {exp}_results.json
PREDS_DIR   = f"{OUTPUT_DIR}/predictions"     # {exp}_results.pkl (has y_true/y_pred/y_proba)

#  Published AAAC benchmark (SVM/MARBERT/AraBERT/DziriBERT — reused, never retrained) 
PUBLISHED_AAAC_RESULTS_DIR = f"{DATASETS_BASE}/AAAC/outputs/results"
PUBLISHED_AAAC_MODELS_DIR  = f"{DATASETS_BASE}/AAAC/outputs/models/adversarial"
PUBLISHED_AAAC_TFIDF_PATH  = f"{DATASETS_BASE}/AAAC/outputs/models/tfidf_vectorizer.joblib"
# NOTE: same paths as configured in 04a — if 04a's Experiment 1 cell loaded successfully there,
# these will resolve correctly here too. If not, fix both notebooks' config cells together.

#  04b outputs (this notebook) 
B_OUT        = f"{OUTPUT_DIR}/04b"
CROSS_DIR    = f"{B_OUT}/cross_dataset"
LANG_DIR     = f"{B_OUT}/language_analysis"
ABLATION_DIR = f"{B_OUT}/ablation"
ERROR_DIR    = f"{B_OUT}/error_analysis"
SUMMARY_DIR  = f"{B_OUT}/summary"
for d in [B_OUT, CROSS_DIR, LANG_DIR, ABLATION_DIR, ERROR_DIR, SUMMARY_DIR]:
    for sub in ["", "/figures", "/tables"]:
        os.makedirs(d + sub, exist_ok=True)

MAX_LENGTH = 64
BATCH_SIZE = 32
LABEL_NAMES = ["benign", "adversarial"]

ALL_MODELS = ["LogReg", "SVM", "MARBERT", "AraBERT", "DziriBERT", "XLM-R"]
CLASSICAL_MODELS = ["LogReg", "SVM"]
TRANSFORMER_MODELS_LIST = ["MARBERT", "AraBERT", "DziriBERT", "XLM-R"]
DOMAINS = ["AAAC", "OSACT", "Unified"]

print("Config loaded. Device:", DEVICE)


## 3. Shared text-cleaning (must match 04a exactly, for consistent inference)

In [ ]:
arabic_diacritics = re.compile(r"[\u064B-\u0652\u0640]")
URL_RE     = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"@\w+")
EMOJI_RE   = re.compile(
    "[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF\u2700-\u27BF]+",
    flags=re.UNICODE
)

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = URL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = EMOJI_RE.sub(" ", text)
    text = re.sub(arabic_diacritics, "", text)
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text); text = re.sub(r"[ؤئ]", "ء", text); text = re.sub(r"ة", "ه", text)
    text = araby.strip_tatweel(text)
    text = text.lower()
    text = re.sub(r"(.)\1{2,}", r"\1", text)
    text = text.replace("_", " ")
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    return re.sub(r"\s+", " ", text).strip()


OSACT_LABEL_MAP = {"safe": 0, "hate_speech": 1}

# bug #6 (still present here even though 04a was fixed): Unified `binary_label` values are
# "safe"/"adversarial", NOT "safe"/"hate_speech" -- must NOT reuse OSACT_LABEL_MAP or every
# "adversarial" row silently maps to NaN and gets dropped by harmonize()'s dropna.
UNIFIED_LABEL_MAP = {"safe": 0, "adversarial": 1}


def harmonize(df, text_col, label_col, label_map=None, variant_col=None,
              default_variant=None, source_name=""):
    out = pd.DataFrame()
    out["text_raw"] = df[text_col].astype(str)
    out["clean_text"] = out["text_raw"].apply(clean_text)
    if label_map is not None:
        out["label"] = df[label_col].map(label_map)
    else:
        out["label"] = df[label_col].astype(int)
    out["language_variant"] = df[variant_col] if variant_col is not None else default_variant
    out["source"] = source_name
    out = out.dropna(subset=["label"]).reset_index(drop=True)
    out["label"] = out["label"].astype(int)
    out = out[out["clean_text"].str.len() > 0].reset_index(drop=True)
    return out


aaac_train_raw = pd.read_csv(AAAC_TRAIN_PATH); aaac_val_raw = pd.read_csv(AAAC_VAL_PATH); aaac_test_raw = pd.read_csv(AAAC_TEST_PATH)
for _df in (aaac_train_raw, aaac_val_raw, aaac_test_raw):
    _df["is_adversarial"] = (_df["category"] != "safe").astype(int)
osact_train_raw = pd.read_csv(OSACT_TRAIN_PATH); osact_dev_raw = pd.read_csv(OSACT_DEV_PATH)
layer1_train_raw = pd.read_csv(LAYER1_TRAIN_PATH); layer1_val_raw = pd.read_csv(LAYER1_VAL_PATH); layer1_test_raw = pd.read_csv(LAYER1_TEST_PATH)

h_aaac_train = harmonize(aaac_train_raw, "text", "is_adversarial", variant_col="language_variant", source_name="aaac")
h_aaac_test  = harmonize(aaac_test_raw,  "text", "is_adversarial", variant_col="language_variant", source_name="aaac")

h_osact_train = harmonize(osact_train_raw, "text", "layer1_label", label_map=OSACT_LABEL_MAP,
                           default_variant="osact_msa", source_name="osact")
h_osact_dev   = harmonize(osact_dev_raw,   "text", "layer1_label", label_map=OSACT_LABEL_MAP,
                           default_variant="osact_msa", source_name="osact")
# reproduce the exact same 90/10 carve-out 04a used, for consistency (deterministic, same seed)
from sklearn.model_selection import train_test_split
osact_train_sub, osact_val_sub = train_test_split(
    h_osact_train, test_size=0.10, stratify=h_osact_train["label"], random_state=SEED
)

h_unified_train = harmonize(layer1_train_raw, "text", "binary_label", label_map=UNIFIED_LABEL_MAP, variant_col="language_variant", source_name="unified")
h_unified_test  = harmonize(layer1_test_raw,  "text", "binary_label", label_map=UNIFIED_LABEL_MAP, variant_col="language_variant", source_name="unified")

assert h_unified_train["label"].nunique() == 2 and h_unified_test["label"].nunique() == 2, (
    "Unified harmonization produced a single-class split -- label map is wrong again, "
    "check UNIFIED_LABEL_MAP vs the raw binary_label values."
)

# Canonical "test set per domain" used throughout this notebook:
TEST_DF = {"AAAC": h_aaac_test, "OSACT": h_osact_dev, "Unified": h_unified_test}
TRAIN_DF = {"AAAC": h_aaac_train, "OSACT": osact_train_sub, "Unified": h_unified_train}

for k, v in TEST_DF.items():
    print(f"{k} test set: {v.shape}")

## 4. Part 1 — Load & Verify Every Required Artifact

Builds a registry of where every (source-domain, model) checkpoint lives, then **fails loudly**
(`FileNotFoundError`) if anything required is missing, rather than silently skipping it.

Registry logic:
- **AAAC** source: SVM / MARBERT / AraBERT / DziriBERT → published checkpoints (`PUBLISHED_AAAC_MODELS_DIR`).
  LogReg / XLM-R → `aaac_transfer` checkpoints trained in 04a §14.
- **OSACT** source: all 6 models → `MODELS_DIR/osact/{model_key}`.
- **Unified** source: all 6 models → `MODELS_DIR/unified/{model_key}`.


In [ ]:
def classical_paths(source, model_key):
    if source == "AAAC" and model_key == "SVM":
        return PUBLISHED_AAAC_TFIDF_PATH, f"{PUBLISHED_AAAC_MODELS_DIR}/SVM/SVM.joblib"
    if source == "AAAC" and model_key == "LogReg":
        return f"{MODELS_DIR}/aaac_transfer/tfidf_vectorizer.joblib", f"{MODELS_DIR}/aaac_transfer/LogReg.joblib"
    exp_key = {"OSACT": "osact", "Unified": "unified"}[source]
    return f"{MODELS_DIR}/{exp_key}/tfidf_vectorizer.joblib", f"{MODELS_DIR}/{exp_key}/{model_key}.joblib"


def transformer_path(source, model_key):
    if source == "AAAC":
        if model_key in ["MARBERT", "AraBERT", "DziriBERT"]:
            return f"{PUBLISHED_AAAC_MODELS_DIR}/{model_key}"
        if model_key == "XLM-R":
            return f"{MODELS_DIR}/aaac_transfer/XLM-R"
    exp_key = {"OSACT": "osact", "Unified": "unified"}[source]
    return f"{MODELS_DIR}/{exp_key}/{model_key}"


REGISTRY = {}
for source in DOMAINS:
    REGISTRY[source] = {}
    for model_key in ALL_MODELS:
        if model_key in CLASSICAL_MODELS:
            tfidf_p, clf_p = classical_paths(source, model_key)
            REGISTRY[source][model_key] = {"kind": "classical", "tfidf": tfidf_p, "model": clf_p}
        else:
            REGISTRY[source][model_key] = {"kind": "transformer", "model": transformer_path(source, model_key)}

missing = []
for source, models in REGISTRY.items():
    for model_key, info in models.items():
        if info["kind"] == "classical":
            for p in [info["tfidf"], info["model"]]:
                if not os.path.exists(p):
                    missing.append(f"[{source}/{model_key}] missing file: {p}")
        else:
            p = info["model"]
            if not os.path.isdir(p) or not any(os.scandir(p)):
                missing.append(f"[{source}/{model_key}] missing/empty model dir: {p}")

if missing:
    raise FileNotFoundError(
        "04b cannot proceed — the following required artifacts from 04a / the published AAAC "
        "benchmark are missing:\n  " + "\n  ".join(missing) +
        "\n\nFix the corresponding path(s) in Section 2 (Config) or re-run the missing part of 04a, "
        "then re-run this cell."
    )
print("All required model artifacts verified present for all", len(DOMAINS), "domains x", len(ALL_MODELS), "models.")


## 5. Generic model loading & inference helpers

In [ ]:
_model_cache = {}

def load_classical(source, model_key):
    key = ("classical", source, model_key)
    if key not in _model_cache:
        info = REGISTRY[source][model_key]
        tfidf = joblib.load(info["tfidf"])
        clf = joblib.load(info["model"])
        _model_cache[key] = (tfidf, clf)
    return _model_cache[key]


def load_transformer(source, model_key):
    key = ("transformer", source, model_key)
    if key not in _model_cache:
        path = REGISTRY[source][model_key]["model"]
        tokenizer = AutoTokenizer.from_pretrained(path)
        model = AutoModelForSequenceClassification.from_pretrained(path).to(DEVICE).eval()
        _model_cache[key] = (tokenizer, model)
    return _model_cache[key]


def predict_classical(source, model_key, texts):
    tfidf, clf = load_classical(source, model_key)
    X = tfidf.transform(texts)
    t0 = time.time()
    y_pred = clf.predict(X)
    pred_time = time.time() - t0
    if hasattr(clf, "predict_proba"):
        y_proba = clf.predict_proba(X)[:, 1]
    else:
        scores = clf.decision_function(X)
        y_proba = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
    return np.array(y_pred), np.array(y_proba), pred_time


@torch.no_grad()
def predict_transformer(source, model_key, texts, batch_size=BATCH_SIZE):
    tokenizer, model = load_transformer(source, model_key)
    texts = list(texts)
    all_preds, all_probas = [], []
    t0 = time.time()
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tokenizer(batch, truncation=True, padding="max_length", max_length=MAX_LENGTH, return_tensors="pt").to(DEVICE)
        logits = model(**enc).logits
        proba = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        pred = np.argmax(logits.cpu().numpy(), axis=1)
        all_preds.append(pred); all_probas.append(proba)
    pred_time = time.time() - t0
    return np.concatenate(all_preds), np.concatenate(all_probas), pred_time


def predict(source, model_key, texts):
    if model_key in CLASSICAL_MODELS:
        return predict_classical(source, model_key, texts)
    return predict_transformer(source, model_key, texts)


print("Inference helpers defined.")


## 6. Extended metrics (Experiment 4's full metric set)

In [ ]:
def extended_metrics(y_true, y_pred, y_proba, pred_time):
    y_true = np.array(y_true); y_pred = np.array(y_pred)
    acc = accuracy_score(y_true, y_pred)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    _, _, weighted_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
    fpr = fp / (fp + tn) if (fp + tn) > 0 else float("nan")
    fnr = fn / (fn + tp) if (fn + tp) > 0 else float("nan")
    try:
        roc_auc = roc_auc_score(y_true, y_proba)
        pr_auc = average_precision_score(y_true, y_proba)
    except Exception:
        roc_auc, pr_auc = None, None
    return {
        "accuracy": acc, "precision": macro_p, "recall": macro_r,
        "macro_precision": macro_p, "macro_recall": macro_r, "macro_f1": macro_f1,
        "weighted_f1": weighted_f1, "roc_auc": roc_auc, "pr_auc": pr_auc,
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "cohen_kappa": cohen_kappa_score(y_true, y_pred),
        "fpr": fpr, "fnr": fnr,
        "confusion_matrix": cm.tolist(),
        "pred_time_s": round(pred_time, 4),
    }


def load_saved_04a(exp_key):
    path = f"{PREDS_DIR}/{exp_key}_results.pkl"
    if not os.path.exists(path):
        return None
    with open(path, "rb") as f:
        return pickle.load(f)


saved_osact    = load_saved_04a("osact")
saved_unified  = load_saved_04a("unified")
saved_transfer = load_saved_04a("aaac_transfer")
saved_aaac_ref = load_saved_04a("aaac_reference")
print("Loaded 04a saved result dicts:",
      {k: (list(v.keys()) if v else None) for k, v in
       {"osact": saved_osact, "unified": saved_unified, "aaac_transfer": saved_transfer, "aaac_reference": saved_aaac_ref}.items()})


## 7. Part 2 — Experiment 4: Cross-Dataset Generalization

7 directions x 6 models. Same-domain (diagonal) cells reuse 04a's already-computed predictions where
available (avoids redundant inference); everything else runs fresh inference using the loaded
checkpoints against the target domain's held-out set — no retraining, ever. Cached diagonal
predictions are length-checked against the current target test set before reuse (same
stale-cache guard as Part 5) and fall back to fresh inference on any mismatch.


In [ ]:
CROSS_PAIRS = [
    ("AAAC", "AAAC"), ("OSACT", "OSACT"), ("Unified", "Unified"),
    ("AAAC", "OSACT"), ("OSACT", "AAAC"),
    ("Unified", "AAAC"), ("Unified", "OSACT"),
]

SAVED_DIAGONAL = {
    ("OSACT", "OSACT"): saved_osact,
    ("Unified", "Unified"): saved_unified,
}

cross_results = {}  # cross_results[(source, target)][model_key] = metrics dict

for source, target in CROSS_PAIRS:
    cross_results[(source, target)] = {}
    target_df = TEST_DF[target]
    y_true = target_df["label"].values
    texts = target_df["clean_text"].values

    for model_key in ALL_MODELS:
        cache_key = f"{source}_to_{target}_{model_key}"
        result_path = f"{CROSS_DIR}/{cache_key}.json"

        reuse_source = None
        if source == "AAAC" and target == "AAAC":
            if model_key in ["MARBERT", "AraBERT", "DziriBERT", "SVM"] and saved_aaac_ref:
                reuse_source = saved_aaac_ref
            elif model_key in ["LogReg", "XLM-R"] and saved_transfer:
                reuse_source = saved_transfer
        elif (source, target) in SAVED_DIAGONAL and SAVED_DIAGONAL[(source, target)]:
            reuse_source = SAVED_DIAGONAL[(source, target)]

        n_expected = len(target_df)
        cache_ok = False
        if reuse_source is not None and model_key in reuse_source and "y_pred" in reuse_source[model_key]:
            r = reuse_source[model_key]
            cached_pred = np.array(r["y_pred"])
            if len(cached_pred) == n_expected:
                cache_ok = True
            else:
                print(f"[WARN] {cache_key}: cached predictions have length {len(cached_pred)}, "
                      f"but current {target} test set has {n_expected} rows -- cache is "
                      f"stale/mismatched (e.g. after the notebook-03 bug #7 OSACT-holdout fix "
                      f"changed test composition). Falling back to fresh inference.")

        if cache_ok:
            r = reuse_source[model_key]
            m = extended_metrics(r["y_true"], r["y_pred"], r.get("y_proba"), r.get("infer_time_s", 0.0))
        else:
            y_pred, y_proba, pred_time = predict(source, model_key, texts)
            m = extended_metrics(y_true, y_pred, y_proba, pred_time)
            assert len(y_pred) == n_expected, (
                f"[{cache_key}] fresh-inference prediction length {len(y_pred)} does not match "
                f"target test length {n_expected} -- investigate the model/checkpoint, not just the cache."
            )

        cross_results[(source, target)][model_key] = m
        with open(result_path, "w") as f:
            json.dump(m, f)

    print(f"  done: {source} -> {target}")

print("\nExperiment 4 complete —", len(cross_results), "direction(s) x", len(ALL_MODELS), "model(s).")


In [ ]:
rows = []
for (source, target), models in cross_results.items():
    for model_key, m in models.items():
        row = {"source": source, "target": target, "model": model_key}
        row.update({k: v for k, v in m.items() if k != "confusion_matrix"})
        rows.append(row)
cross_df = pd.DataFrame(rows)
cross_df.to_csv(f"{CROSS_DIR}/tables/experiment4_cross_dataset_full.csv", index=False)

cross_matrix = cross_df.pivot_table(index=["source", "target"], columns="model", values="macro_f1")
cross_matrix.to_csv(f"{CROSS_DIR}/tables/experiment4_macro_f1_matrix.csv")
display(cross_matrix.style.background_gradient(cmap="Blues", axis=None).format("{:.3f}"))


In [ ]:
# Confusion matrices for every direction/model, saved 300dpi
for (source, target), models in cross_results.items():
    for model_key, m in models.items():
        cm = np.array(m["confusion_matrix"])
        fig, ax = plt.subplots(figsize=(3.2, 2.8))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                    xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax)
        ax.set_title(f"{source} -> {target}: {model_key}", fontsize=9)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        plt.tight_layout()
        fig.savefig(f"{CROSS_DIR}/figures/cm_{source}_to_{target}_{model_key}.png", dpi=300)
        plt.close(fig)
print("Saved", sum(len(v) for v in cross_results.values()), "confusion-matrix figures to", f"{CROSS_DIR}/figures")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
heat_df = cross_df.copy()
heat_df["direction"] = heat_df["source"] + " -> " + heat_df["target"]
pivot = heat_df.pivot_table(index="direction", columns="model", values="macro_f1")
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="viridis", ax=ax)
ax.set_title("Experiment 4 — Cross-Dataset Macro F1")
plt.tight_layout()
fig.savefig(f"{CROSS_DIR}/figures/experiment4_heatmap.png", dpi=300)
fig.savefig(f"{CROSS_DIR}/figures/experiment4_heatmap.pdf")
plt.show()


## 8. Part 3 — Experiment 5: Language-wise Robustness

AAAC and Unified carry `language_variant` (Darja / Arabizi / Franco-Arabic, plus `osact_msa` for
OSACT-origin rows in Unified). OSACT alone has no script-variant column — it is reported as a single
monolithic Arabic-script/MSA group, not split further.

Per-variant metrics use the **matched in-domain model** (model trained on that same domain, evaluated
on its own test set, filtered by variant) — e.g. AAAC-trained models on AAAC test filtered by variant,
Unified-trained models on Unified test filtered by variant. Small-n groups (< 30 samples) additionally
report a Wilson 95% CI instead of a bare percentage.


In [ ]:
def wilson_ci(successes, n, alpha=0.05):
    if n == 0:
        return (float("nan"), float("nan"))
    lo, hi = proportion_confint(successes, n, alpha=alpha, method="wilson")
    return lo, hi


def language_breakdown(domain, model_key):
    df = TEST_DF[domain]
    if source_reuse := (domain == "OSACT"):
        pass  # OSACT has only one variant tag; handled generically below anyway
    y_true_full = df["label"].values
    texts_full = df["clean_text"].values

    reuse_source = None
    if domain == "AAAC" and model_key in ["MARBERT", "AraBERT", "DziriBERT", "SVM"] and saved_aaac_ref:
        reuse_source = saved_aaac_ref
    elif domain == "AAAC" and model_key in ["LogReg", "XLM-R"] and saved_transfer:
        reuse_source = saved_transfer
    elif domain == "OSACT" and saved_osact:
        reuse_source = saved_osact
    elif domain == "Unified" and saved_unified:
        reuse_source = saved_unified

    n_expected = len(df)
    y_pred_full, y_proba_full = None, None
    if reuse_source is not None and model_key in reuse_source and "y_pred" in reuse_source[model_key]:
        cached_pred = np.array(reuse_source[model_key]["y_pred"])
        if len(cached_pred) == n_expected:
            y_pred_full = cached_pred
            cached_proba = reuse_source[model_key].get("y_proba")
            y_proba_full = np.array(cached_proba) if cached_proba is not None else np.array([np.nan] * n_expected)
        else:
            print(f"[WARN] {domain}/{model_key} language_breakdown: cached predictions have length "
                  f"{len(cached_pred)}, but current {domain} test set has {n_expected} rows -- "
                  f"cache is stale/mismatched. Falling back to fresh inference.")
    if y_pred_full is None:
        y_pred_full, y_proba_full, _ = predict(domain, model_key, texts_full)
        assert len(y_pred_full) == n_expected, (
            f"[{domain}/{model_key}] fresh-inference prediction length {len(y_pred_full)} still "
            f"does not match {domain} test length {n_expected} -- investigate the model/checkpoint."
        )

    rows = []
    for variant, sub in df.groupby("language_variant"):
        idx = sub.index.values
        yt, yp = y_true_full[idx], y_pred_full[idx]
        n = len(yt)
        acc = accuracy_score(yt, yp) if n else float("nan")
        p, r, f1, _ = precision_recall_fscore_support(yt, yp, average="macro", zero_division=0) if n else (np.nan,) * 3 + (None,)
        _, _, wf1, _ = precision_recall_fscore_support(yt, yp, average="weighted", zero_division=0) if n else (np.nan,) * 3 + (None,)
        cm = confusion_matrix(yt, yp, labels=[0, 1]) if n else np.zeros((2, 2))
        tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
        fpr = fp / (fp + tn) if (fp + tn) > 0 else float("nan")
        fnr = fn / (fn + tp) if (fn + tp) > 0 else float("nan")
        row = {
            "domain": domain, "model": model_key, "language_variant": variant, "n": n,
            "accuracy": acc, "precision": p, "recall": r, "macro_f1": f1, "weighted_f1": wf1,
            "fpr": fpr, "fnr": fnr,
        }
        if n < 30:
            correct = int(np.sum(yt == yp))
            lo, hi = wilson_ci(correct, n)
            row["wilson_ci_low"] = lo
            row["wilson_ci_high"] = hi
        rows.append(row)
    return rows


lang_rows = []
for domain in DOMAINS:
    for model_key in ALL_MODELS:
        lang_rows.extend(language_breakdown(domain, model_key))
lang_df = pd.DataFrame(lang_rows)
lang_df.to_csv(f"{LANG_DIR}/tables/experiment5_language_breakdown.csv", index=False)
display(lang_df.sort_values(["domain", "model", "language_variant"]))


In [ ]:
for domain in DOMAINS:
    sub = lang_df[lang_df["domain"] == domain]
    if sub["language_variant"].nunique() <= 1:
        print(f"[{domain}] only one language_variant group — skipping bar plot (nothing to compare).")
        continue
    fig, ax = plt.subplots(figsize=(8, 4.5))
    sns.barplot(data=sub, x="language_variant", y="macro_f1", hue="model", ax=ax)
    ax.set_title(f"Experiment 5 — Per-Script Macro F1 ({domain})")
    ax.set_ylim(0, 1)
    plt.xticks(rotation=20)
    plt.tight_layout()
    fig.savefig(f"{LANG_DIR}/figures/experiment5_{domain}_bars.png", dpi=300)
    fig.savefig(f"{LANG_DIR}/figures/experiment5_{domain}_bars.pdf")
    plt.show()


## 9. Part 4 — Experiment 9: Ablation Study (AAAC-only vs OSACT-only vs Unified)

Reuses the diagonal cells from Experiment 4 (`AAAC->AAAC`, `OSACT->OSACT`, `Unified->Unified`) as the
"trained and evaluated on X" numbers per model, then computes absolute / relative / percentage
improvement of Unified over each single-source baseline, with best/second-best/worst auto-highlighted
per metric.


In [ ]:
ABLATION_METRICS = ["macro_f1", "weighted_f1", "roc_auc", "pr_auc", "fpr", "fnr", "balanced_accuracy"]

ablation_rows = []
for model_key in ALL_MODELS:
    aaac_m    = cross_results[("AAAC", "AAAC")][model_key]
    osact_m   = cross_results[("OSACT", "OSACT")][model_key]
    unified_m = cross_results[("Unified", "Unified")][model_key]
    row = {"model": model_key}
    for metric in ABLATION_METRICS:
        a, o, u = aaac_m.get(metric), osact_m.get(metric), unified_m.get(metric)
        row[f"AAAC_only_{metric}"] = a
        row[f"OSACT_only_{metric}"] = o
        row[f"Unified_{metric}"] = u
        lower_is_better = metric in ("fpr", "fnr")
        for base_name, base_val in [("AAAC", a), ("OSACT", o)]:
            if base_val is None or u is None:
                continue
            abs_imp = (base_val - u) if lower_is_better else (u - base_val)
            rel_imp = abs_imp / base_val if base_val not in (0, None) else float("nan")
            row[f"abs_improvement_vs_{base_name}_{metric}"] = abs_imp
            row[f"pct_improvement_vs_{base_name}_{metric}"] = rel_imp * 100
    ablation_rows.append(row)

ablation_df = pd.DataFrame(ablation_rows)


def highlight_best_worst(df, metric_cols, higher_is_better=True):
    ranks = df[metric_cols].rank(ascending=not higher_is_better, method="min")
    best = ranks.eq(1.0).apply(lambda col: df.loc[col, "model"].tolist() if col.any() else [], axis=0)
    return ranks


for metric in ABLATION_METRICS:
    col = f"Unified_{metric}"
    if col not in ablation_df:
        continue
    higher_is_better = metric not in ("fpr", "fnr")
    order = ablation_df[col].rank(ascending=not higher_is_better, method="min")
    ablation_df[f"{metric}_rank"] = order
    best_model = ablation_df.loc[order.idxmin(), "model"]
    worst_model = ablation_df.loc[order.idxmax(), "model"]
    print(f"[Unified {metric}] best: {best_model} | worst: {worst_model}")

ablation_df.to_csv(f"{ABLATION_DIR}/tables/experiment9_ablation_full.csv", index=False)
try:
    ablation_df.to_excel(f"{ABLATION_DIR}/tables/experiment9_ablation_full.xlsx", index=False)
except Exception as e:
    print("[WARN] xlsx export skipped:", e)
with open(f"{ABLATION_DIR}/tables/experiment9_ablation_full.tex", "w") as f:
    f.write(ablation_df.to_latex(index=False, float_format="%.4f"))
with open(f"{ABLATION_DIR}/tables/experiment9_ablation_full.md", "w") as f:
    f.write(ablation_df.to_markdown(index=False))

display(ablation_df[["model"] + [f"Unified_{m}" for m in ABLATION_METRICS] +
                     [f"abs_improvement_vs_AAAC_{m}" for m in ABLATION_METRICS]])


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_df = ablation_df.melt(id_vars="model",
                            value_vars=[f"AAAC_only_macro_f1", f"OSACT_only_macro_f1", f"Unified_macro_f1"],
                            var_name="condition", value_name="macro_f1")
plot_df["condition"] = plot_df["condition"].str.replace("_macro_f1", "")
sns.barplot(data=plot_df, x="model", y="macro_f1", hue="condition", ax=ax)
ax.set_title("Experiment 9 — Ablation: Macro F1 by Training Source")
ax.set_ylim(0, 1)
plt.tight_layout()
fig.savefig(f"{ABLATION_DIR}/figures/experiment9_ablation_bars.png", dpi=300)
fig.savefig(f"{ABLATION_DIR}/figures/experiment9_ablation_bars.pdf")
plt.show()


## 10. Part 5 — Error Analysis

Primary basis: **Unified test set**, evaluated with each model's Unified-trained checkpoint (the most
complete, largest, and directly relevant deployment scenario for Layer 1). For every transformer we
extract TP/TN/FP/FN, then compute cross-model agreement sets (correct-by-all, missed-by-all,
correct-only-by-model-X). Every FP/FN is exported with text, gold label, prediction, probability,
dataset, language variant, and token length.


In [ ]:
n_expected = len(h_unified_test)
unified_preds = {}
for model_key in ALL_MODELS:
    y_pred, y_proba = None, None
    if saved_unified and model_key in saved_unified and "y_pred" in saved_unified[model_key]:
        cached_pred = np.array(saved_unified[model_key]["y_pred"])
        if len(cached_pred) == n_expected:
            y_pred = cached_pred
            cached_proba = saved_unified[model_key].get("y_proba")
            y_proba = np.array(cached_proba) if cached_proba is not None else np.array([np.nan] * n_expected)
        else:
            print(f"[WARN] cached '{model_key}' predictions have length {len(cached_pred)}, "
                  f"but current h_unified_test has {n_expected} rows — cache is stale/mismatched. "
                  f"Falling back to fresh inference instead of trusting it.")
    if y_pred is None:
        y_pred, y_proba, _ = predict("Unified", model_key, h_unified_test["clean_text"].values)
    assert len(y_pred) == n_expected, (
        f"[{model_key}] prediction length {len(y_pred)} still does not match h_unified_test "
        f"length {n_expected} even after fresh inference — investigate the model/checkpoint, "
        f"not just the cache."
    )
    unified_preds[model_key] = (y_pred, y_proba)

y_true = h_unified_test["label"].values
correct_masks = {mk: (unified_preds[mk][0] == y_true) for mk in ALL_MODELS}

correct_by_all = np.all(list(correct_masks.values()), axis=0)
missed_by_all  = np.all([~m for m in correct_masks.values()], axis=0)

only_correct = {}
for mk in ALL_MODELS:
    others_wrong = np.all([~correct_masks[o] for o in ALL_MODELS if o != mk], axis=0)
    only_correct[mk] = correct_masks[mk] & others_wrong

print("Correct by ALL models       :", int(correct_by_all.sum()))
print("Missed by ALL models        :", int(missed_by_all.sum()))
for mk in ALL_MODELS:
    print(f"Only {mk} got it right      :", int(only_correct[mk].sum()))

agreement_summary = pd.DataFrame({
    "category": ["correct_by_all", "missed_by_all"] + [f"only_{mk}_correct" for mk in ALL_MODELS],
    "count": [int(correct_by_all.sum()), int(missed_by_all.sum())] +
             [int(only_correct[mk].sum()) for mk in ALL_MODELS],
})
agreement_summary.to_csv(f"{ERROR_DIR}/tables/agreement_summary.csv", index=False)
display(agreement_summary)

In [ ]:
def export_examples(mask, name):
    idx = np.where(mask)[0]
    sub = h_unified_test.iloc[idx].copy()
    sub["gold_label"] = y_true[idx]
    for mk in ALL_MODELS:
        sub[f"pred_{mk}"] = unified_preds[mk][0][idx]
        sub[f"proba_{mk}"] = unified_preds[mk][1][idx]
    sub["token_length"] = sub["clean_text"].str.split().apply(len)
    sub = sub.rename(columns={"text_raw": "text", "source": "dataset", "language_variant": "language"})
    cols = ["text", "gold_label", "dataset", "language", "token_length"] + \
           [c for c in sub.columns if c.startswith("pred_") or c.startswith("proba_")]
    sub[cols].to_csv(f"{ERROR_DIR}/{name}.csv", index=False)
    print(f"Saved {len(sub)} examples -> {ERROR_DIR}/{name}.csv")

export_examples(correct_by_all, "correct_by_all_models")
export_examples(missed_by_all, "missed_by_all_models")
for mk in ALL_MODELS:
    export_examples(only_correct[mk], f"only_{mk}_correct")


In [ ]:
for model_key in TRANSFORMER_MODELS_LIST:
    y_pred, y_proba = unified_preds[model_key]
    fp_mask = (y_pred == 1) & (y_true == 0)
    fn_mask = (y_pred == 0) & (y_true == 1)
    for mask, tag in [(fp_mask, "false_positives"), (fn_mask, "false_negatives")]:
        idx = np.where(mask)[0]
        sub = h_unified_test.iloc[idx].copy()
        sub["prediction"] = y_pred[idx]
        sub["probability"] = y_proba[idx]
        sub["token_length"] = sub["clean_text"].str.split().apply(len)
        sub = sub.rename(columns={"text_raw": "text", "label": "gold_label",
                                   "source": "dataset", "language_variant": "language"})
        out_cols = ["text", "gold_label", "prediction", "probability", "dataset", "language", "token_length"]
        sub[out_cols].to_csv(f"{ERROR_DIR}/{model_key}_{tag}.csv", index=False)
        print(f"[{model_key}] {tag}: {len(sub)} rows -> {ERROR_DIR}/{model_key}_{tag}.csv")


## 11. Part 6 — Dataset Transfer / Domain-Shift Analysis

Quantifies how different AAAC, OSACT, and Unified actually are as text distributions: length, token
length, vocabulary size, OOV rate (relative to the Unified vocabulary), label distribution, language
distribution, TF-IDF cosine similarity between corpus centroids, Jensen-Shannon / KL divergence of
label distributions, and lexical (vocabulary) overlap.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.spatial.distance import jensenshannon
from scipy.stats import entropy

domain_texts = {
    "AAAC": h_aaac_train["clean_text"],
    "OSACT": h_osact_train["clean_text"],
    "Unified": h_unified_train["clean_text"],
}
domain_labels = {
    "AAAC": h_aaac_train["label"],
    "OSACT": h_osact_train["label"],
    "Unified": h_unified_train["label"],
}

transfer_rows = []
vocabs = {}
for name, texts in domain_texts.items():
    lengths_chars = texts.str.len()
    lengths_tokens = texts.str.split().apply(len)
    vocab = set(w for t in texts for w in t.split())
    vocabs[name] = vocab
    label_counts = domain_labels[name].value_counts(normalize=True).to_dict()
    transfer_rows.append({
        "dataset": name, "n_rows": len(texts),
        "avg_char_length": lengths_chars.mean(), "avg_token_length": lengths_tokens.mean(),
        "vocab_size": len(vocab),
        "pct_label_0": label_counts.get(0, 0.0) * 100, "pct_label_1": label_counts.get(1, 0.0) * 100,
    })
transfer_df = pd.DataFrame(transfer_rows)

unified_vocab = vocabs["Unified"]
for name in ["AAAC", "OSACT"]:
    oov = len(vocabs[name] - unified_vocab) / max(len(vocabs[name]), 1)
    transfer_df.loc[transfer_df["dataset"] == name, "oov_rate_vs_unified"] = oov

    overlap = len(vocabs[name] & unified_vocab) / max(len(vocabs[name] | unified_vocab), 1)
    transfer_df.loc[transfer_df["dataset"] == name, "lexical_overlap_vs_unified"] = overlap

# TF-IDF centroid cosine similarity across all three corpora
tfidf_all = TfidfVectorizer(max_features=20000)
all_texts = pd.concat([domain_texts["AAAC"], domain_texts["OSACT"], domain_texts["Unified"]])
tfidf_all.fit(all_texts)
centroids = {name: np.asarray(tfidf_all.transform(texts).mean(axis=0)).ravel()
             for name, texts in domain_texts.items()}
from sklearn.metrics.pairwise import cosine_similarity
pairs = [("AAAC", "OSACT"), ("AAAC", "Unified"), ("OSACT", "Unified")]
cos_rows = []
for a, b in pairs:
    sim = cosine_similarity([centroids[a]], [centroids[b]])[0, 0]
    p_a = domain_labels[a].value_counts(normalize=True).reindex([0, 1], fill_value=1e-9).values
    p_b = domain_labels[b].value_counts(normalize=True).reindex([0, 1], fill_value=1e-9).values
    js = jensenshannon(p_a, p_b) ** 2
    kl = entropy(p_a, p_b)
    cos_rows.append({"pair": f"{a} vs {b}", "tfidf_cosine_similarity": sim,
                      "label_js_divergence": js, "label_kl_divergence": kl})
cos_df = pd.DataFrame(cos_rows)

transfer_df.to_csv(f"{SUMMARY_DIR}/tables/dataset_transfer_stats.csv", index=False)
cos_df.to_csv(f"{SUMMARY_DIR}/tables/dataset_transfer_similarity.csv", index=False)
display(transfer_df)
display(cos_df)


## 12. Part 8 — Publication Figures (radar chart, ROC/PR overlays, performance vs. inference time)

In [ ]:
def radar_chart(model_scores, metrics, title, save_path):
    n = len(metrics)
    angles = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
    angles += angles[:1]
    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
    for model_key, scores in model_scores.items():
        vals = [scores.get(m, 0) or 0 for m in metrics]
        vals += vals[:1]
        ax.plot(angles, vals, label=model_key)
        ax.fill(angles, vals, alpha=0.05)
    ax.set_xticks(angles[:-1]); ax.set_xticklabels(metrics)
    ax.set_title(title)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=8)
    plt.tight_layout()
    fig.savefig(save_path, dpi=300)
    plt.show()


radar_metrics = ["macro_f1", "weighted_f1", "roc_auc", "pr_auc", "balanced_accuracy"]
radar_scores = {mk: cross_results[("Unified", "Unified")][mk] for mk in ALL_MODELS}
radar_chart(radar_scores, radar_metrics, "Model Comparison (Unified, in-domain)",
            f"{SUMMARY_DIR}/figures/radar_unified.png")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for model_key in ALL_MODELS:
    if model_key in CLASSICAL_MODELS:
        tfidf, clf = load_classical("Unified", model_key)
        X = tfidf.transform(h_unified_test["clean_text"])
        proba = clf.predict_proba(X)[:, 1] if hasattr(clf, "predict_proba") else clf.decision_function(X)
    else:
        _, proba, _ = predict_transformer("Unified", model_key, h_unified_test["clean_text"].values)
    fpr_curve, tpr_curve, _ = roc_curve(h_unified_test["label"].values, proba)
    ax.plot(fpr_curve, tpr_curve, label=model_key)
ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — Unified test set")
ax.legend()
plt.tight_layout()
fig.savefig(f"{SUMMARY_DIR}/figures/roc_curves_unified.png", dpi=300)
fig.savefig(f"{SUMMARY_DIR}/figures/roc_curves_unified.pdf")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for model_key in ALL_MODELS:
    if model_key in CLASSICAL_MODELS:
        tfidf, clf = load_classical("Unified", model_key)
        X = tfidf.transform(h_unified_test["clean_text"])
        proba = clf.predict_proba(X)[:, 1] if hasattr(clf, "predict_proba") else clf.decision_function(X)
    else:
        _, proba, _ = predict_transformer("Unified", model_key, h_unified_test["clean_text"].values)
    prec_curve, rec_curve, _ = precision_recall_curve(h_unified_test["label"].values, proba)
    ax.plot(rec_curve, prec_curve, label=model_key)
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves — Unified test set")
ax.legend()
plt.tight_layout()
fig.savefig(f"{SUMMARY_DIR}/figures/pr_curves_unified.png", dpi=300)
fig.savefig(f"{SUMMARY_DIR}/figures/pr_curves_unified.pdf")
plt.show()


In [ ]:
perf_time_rows = []
for model_key in ALL_MODELS:
    m = cross_results[("Unified", "Unified")][model_key]
    perf_time_rows.append({"model": model_key, "macro_f1": m["macro_f1"], "pred_time_s": m["pred_time_s"]})
pt_df = pd.DataFrame(perf_time_rows)

fig, ax = plt.subplots(figsize=(6.5, 5))
sns.scatterplot(data=pt_df, x="pred_time_s", y="macro_f1", hue="model", s=140, ax=ax)
for _, r in pt_df.iterrows():
    ax.annotate(r["model"], (r["pred_time_s"], r["macro_f1"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
ax.set_xscale("log")
ax.set_xlabel("Inference time (s, log scale)"); ax.set_ylabel("Macro F1")
ax.set_title("Performance vs. Inference Time (Unified test set)")
plt.tight_layout()
fig.savefig(f"{SUMMARY_DIR}/figures/performance_vs_time.png", dpi=300)
fig.savefig(f"{SUMMARY_DIR}/figures/performance_vs_time.pdf")
plt.show()


## 13. Part 9 — Research Report (`04b_research_report.csv`)

Flattens every experiment's results into one long-format table, with a handful of rule-based
`observation` flags (e.g. domain-transfer gaps > 10 Macro-F1 points) ready for thesis insertion.


In [ ]:
report_rows = []

for (source, target), models in cross_results.items():
    for model_key, m in models.items():
        report_rows.append({
            "experiment": "Experiment 4 - Cross-Dataset Generalization",
            "source": source, "target": target, "model": model_key,
            "metric": "macro_f1", "value": m["macro_f1"],
        })

for _, row in lang_df.iterrows():
    report_rows.append({
        "experiment": "Experiment 5 - Language-wise Robustness",
        "source": row["domain"], "target": row["language_variant"], "model": row["model"],
        "metric": "macro_f1", "value": row["macro_f1"],
    })

for _, row in ablation_df.iterrows():
    for metric in ABLATION_METRICS:
        report_rows.append({
            "experiment": "Experiment 9 - Ablation",
            "source": "Unified", "target": "Unified", "model": row["model"],
            "metric": metric, "value": row.get(f"Unified_{metric}"),
        })

report_df = pd.DataFrame(report_rows)


def auto_observation(row):
    if row["experiment"].startswith("Experiment 4") and row["source"] != row["target"] and row["metric"] == "macro_f1":
        diag_key = (row["source"], row["source"])
        if diag_key in cross_results and row["model"] in cross_results[diag_key]:
            in_domain = cross_results[diag_key][row["model"]]["macro_f1"]
            gap = in_domain - row["value"]
            if gap > 0.10:
                return f"Domain-transfer gap of {gap:.3f} Macro F1 vs in-domain — notable generalization drop."
    return ""


report_df["observation"] = report_df.apply(auto_observation, axis=1)
report_df.to_csv(f"{B_OUT}/04b_research_report.csv", index=False)
print("Saved research report:", f"{B_OUT}/04b_research_report.csv", "-", len(report_df), "rows")
display(report_df.head(20))
